# Rotulagem por similaridade

### Extração de clusters

In [1]:
import os

import hdbscan
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

os.environ["TOKENIZERS_PARALLELISM"] = "false"

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")


umap_model = UMAP(n_neighbors=17, n_components=8, metric='cosine')


hdbscan_model = hdbscan.HDBSCAN(min_cluster_size = 60,
                                metric='euclidean', 
                                cluster_selection_method='eom')


vectorizer_model = CountVectorizer(ngram_range=(2, 2), stop_words="english")



topic_model = BERTopic(embedding_model=model,
                       top_n_words=10,
                       nr_topics = 'auto',
                       umap_model=umap_model,
                       hdbscan_model=hdbscan_model,
                       vectorizer_model=vectorizer_model,
                       low_memory=True,
                       calculate_probabilities=False, 
                       verbose=True)

/home/andre/Documentos/Workspace/mestrado/dissertacao/venv/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

docs = pd.read_csv('../data/processed/trump_processed.csv')

topics, probs = topic_model.fit_transform(docs.text)

Batches: 100%|██████████| 9148/9148 [00:43<00:00, 209.99it/s]
2022-12-22 08:49:54,532 - BERTopic - Transformed documents to Embeddings


In [ ]:
topic_model.visualize_topics()

### Buscar os clusters correspondentes a cada linha do query set e atribuir-los a label correspondente

- Desafios
    - O mesmo tópico pode ser atrbuído a rótulos diferentes dependendo da consulta
        - Criar sistema para atribuir ao tópico o rótulo mais frequênte


In [ ]:
import pandas as pd

query_set = pd.read_csv('../data/validation.csv')

In [ ]:
def find_clusters(query_set):
    clusters_list = list()
    for index in range(len(query_set)):
        finded_clusters = topic_model.find_topics(query_set.text[index])
        for cluster_number in finded_clusters[0]:
            if cluster_number != -1:
                assigned_clusters = {
                    "number": cluster_number,
                    "label" : query_set.target[index]
                }
                clusters_list.append(assigned_clusters)

    return pd.DataFrame(clusters_list)

pre_labeled_clusters = find_clusters(query_set)

In [ ]:
pre_labeled_clusters[pre_labeled_clusters.number == 47]

In [ ]:
import numpy as np

def find_most_frequent_label(pre_labeled_clusters):
    clusters_list = list()
    for topic_number in pre_labeled_clusters.number.unique():
        grouped_topics = pre_labeled_clusters[pre_labeled_clusters.number == topic_number].groupby(['number','label']).label.count()
        most_frequent_label = {
            'number': topic_number,
            'label' : grouped_topics.index[np.argmax(grouped_topics)][1]
        }
        clusters_list.append(most_frequent_label)
    return pd.DataFrame(clusters_list)
    
labeled_clusters = find_most_frequent_label(pre_labeled_clusters)

In [ ]:
labeled_clusters[labeled_clusters.number == 47].label.values[0]

### Rotular  os documentos presentes nos clusters com as labels correspondente

In [ ]:
topic_docs = {topic: [] for topic in set(topics)}
for topic, doc in zip(topics, docs.text):
    topic_docs[topic].append(doc)

In [ ]:
docs_list = list()
for cluster_number in labeled_clusters.number:
   for docs in topic_docs[cluster_number]:
                docs ={
                    "text": docs,
                    "label": labeled_clusters[labeled_clusters.number == cluster_number].label.values[0]
                }
                docs_list.append(docs)

In [ ]:
labeled_dataset = pd.DataFrame(docs_list)
labeled_dataset

In [ ]:
labeled_dataset.to_csv('../data/labeled_dataset_multi-qa-MiniLM-L6-cos-v1.csv', index=False)

In [ ]:
import seaborn as sns

sns.countplot(x=labeled_dataset['label'], label = 'count')